In [ ]:
import pandas as pd
from pathlib import Path

def export_to_spreadsheet(export_folder_path, filename, data):
    # Create the full path to the export file
    export_file_path = Path(export_folder_path) / filename

    # Create the spreadsheet
    with pd.ExcelWriter(export_file_path.absolute(), engine="openpyxl") as writer:
        # The input is a dictionary of data frames - the key is the required name of the worksheet in the
        # final spreadsheet and the value is the data to be written to that sheet
        for key, value in data.items():
            if value is not None:
                # Excel cannot store timezone-aware datetimes, so export an equivalent timezone-naive copy.
                export_value = value.copy()
                for column in export_value.select_dtypes(include=['datetimetz']).columns:
                    export_value[column] = export_value[column].dt.tz_localize(None)
                export_value.to_excel(writer, sheet_name=key, index=False)

In [ ]:
import pandas as pd
from pathlib import Path

def export_to_csv(export_folder_path, filename, data):
    # Create the full path to the export file
    export_file_path = Path(export_folder_path) / filename

    # Create the CSV file
    data.to_csv(export_file_path.absolute(), index=False)

In [ ]:
from matplotlib.figure import Figure

SUPPORTED_CHART_FORMATS = {"png", "pdf"}
CHART_EXPORT_DPI = 300

def export_chart(
    export_folder_path: str | Path,
    name: str,
    extension: str,
    figure: Figure,
) -> Path:
    """Export a specific chart figure in a supported format.

    :param export_folder_path: Folder in which to create the chart file.
    :param name: Filename without an extension.
    :param extension: Output format, either PNG or PDF.
    :param figure: Matplotlib figure displayed by the reporting notebook.
    :return: Absolute path to the exported chart.
    :raises ValueError: If the requested output format is unsupported.
    """
    chart_format = extension.casefold()
    if chart_format not in SUPPORTED_CHART_FORMATS:
        raise ValueError(f"Unsupported chart format: {extension}")

    chart_file_path = Path(export_folder_path) / f"{name}.{chart_format}"
    figure.savefig(
        chart_file_path.absolute(),
        format=chart_format,
        dpi=CHART_EXPORT_DPI,
        bbox_inches="tight",
    )
    return chart_file_path.absolute()
